# XGBoost Regression for Exam Score Prediction

This notebook trains an XGBoost regressor to predict a student's exam score using the available study and academic features.
Each code cell includes comments to explain the purpose of the step and the logic used.

In [ ]:
# Step 1: Import the modules required for data loading, preprocessing, and modeling.
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

RANDOM_STATE = 42


In [ ]:
# Step 2: Load the dataset from the repository and inspect the first rows.
project_root = Path.cwd()
data_candidates = [
    project_root / 'data loading' / 'Exam_Score_Prediction.csv',
    project_root.parent / 'data loading' / 'Exam_Score_Prediction.csv',
]
data_path = next((candidate for candidate in data_candidates if candidate.exists()), data_candidates[0])
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
df.head()


In [ ]:
# Step 3: Check for missing values to confirm the data is ready for modeling.
missing_values = df.isna().sum()
print(missing_values[missing_values > 0])

# If there are no missing values, this confirms we can proceed without extra cleanup.
df.info()


## Feature and target selection

We separate the target variable, `exam_score`, from the input features. Categorical features are encoded automatically, while numerical features are handled with median imputation.

In [ ]:
# Step 4: Define the target and drop the non-feature columns from the input matrix.
target_col = 'exam_score'
features = df.drop(columns=['student_id', target_col])
target = df[target_col]

# Show the first few feature names so the model input is clearly understood.
print('Feature columns:', list(features.columns))
print(f'Target column: {target_col}')


In [ ]:
# Step 5: Split the data into training and test sets for honest evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f'Training rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')


In [ ]:
# Step 6: Create separate preprocessing pipelines for numeric and categorical fields.
numeric_features = features.select_dtypes(exclude=['object']).columns.tolist()
categorical_features = features.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features),
    ]
)

print('Numeric columns:', numeric_features)
print('Categorical columns:', categorical_features)


In [ ]:
# Step 7: Build the XGBoost regressor and wrap it in a pipeline with the preprocessor.
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model),
    ]
)

pipeline.fit(X_train, y_train)
print('Model training complete.')


In [ ]:
# Step 8: Evaluate the model on the unseen test set using standard regression metrics.
predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)

print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R^2 Score: {r2:.4f}')


In [ ]:
# Step 9: Inspect a few sample predictions to confirm that the model produces realistic values.
sample = X_test.head(5)
sample_predictions = pipeline.predict(sample)

for idx, value in enumerate(sample_predictions):
    actual = y_test.iloc[idx]
    print(f'Row {idx}: predicted = {value:.2f}, actual = {actual:.2f}')


## Final note

The XGBoost regressor is a strong choice for tabular data because it handles non-linear relationships well and uses tree-based boosting to improve prediction quality. The metrics above help determine whether the model is accurate enough for the task.

## Manual testing

Use this section to provide a new student profile manually and predict the exam score with the trained XGBoost model.

In [ ]:
# Step 10: Test the trained model with a manually entered student record.
manual_student = {
    'age': 22,
    'gender': 'male',
    'course': 'b.tech',
    'study_hours': 6.5,
    'class_attendance': 88.0,
    'internet_access': 'yes',
    'sleep_hours': 7.5,
    'sleep_quality': 'good',
    'study_method': 'self-study',
    'facility_rating': 'high',
    'exam_difficulty': 'moderate',
}

manual_df = pd.DataFrame([manual_student])
predicted_score = pipeline.predict(manual_df)[0]

print(f'Manual test input: {manual_student}')
print(f'Predicted exam score: {predicted_score:.2f}')
